### RAG application using Type Sense

In [8]:
import typesense

In [41]:
import os
from dotenv import load_dotenv

load_dotenv("notebook/.env")

client = typesense.Client({
    'nodes': [{
        'host': 'kqzi1b2675vwo8h3p-1.a1.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': os.getenv("TYPESENSE_API_KEY"),
    'connection_timeout_seconds': 2
})

books_schema = {
    'name': 'books',

    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'author', 'type': 'string', 'facet': True},
        {'name': 'country', 'type': 'string', 'facet': True},
        {'name': 'language', 'type': 'string', 'facet': True},
        {'name': 'pages', 'type': 'int32'},
        {'name': 'year', 'type': 'int32'},
        {'name': 'link', 'type': 'string'},
        {'name': 'imageLink', 'type': 'string'}
    ]
}
collection = client.collections['books']

In [11]:
client

In [14]:
import json

# Read JSON file
with open("books.json", "r", encoding="utf-8") as f:
    books = json.load(f)

# Create JSONL file
with open("books.jsonl", "w", encoding="utf-8") as f:
    for i, book in enumerate(books, start=1):
        book["id"] = str(i)
        f.write(json.dumps(book, ensure_ascii=False) + "\n")

print("JSON converted to JSONL successfully!")
print("Number of books:", len(books))

JSON converted to JSONL successfully!
Number of books: 100


In [15]:
with open("books.jsonl", "r", encoding="utf-8") as f:
    print(f.readline())

{"author": "Chinua Achebe", "country": "Nigeria", "imageLink": "images/things-fall-apart.jpg", "language": "English", "link": "https://en.wikipedia.org/wiki/Things_Fall_Apart\n", "pages": 209, "title": "Things Fall Apart", "year": 1958, "id": "1"}



In [16]:
with open("books.jsonl", "r", encoding="utf-8") as f:
    print(f.readline())
    print(f.readline())

{"author": "Chinua Achebe", "country": "Nigeria", "imageLink": "images/things-fall-apart.jpg", "language": "English", "link": "https://en.wikipedia.org/wiki/Things_Fall_Apart\n", "pages": 209, "title": "Things Fall Apart", "year": 1958, "id": "1"}

{"author": "Hans Christian Andersen", "country": "Denmark", "imageLink": "images/fairy-tales.jpg", "language": "Danish", "link": "https://en.wikipedia.org/wiki/Fairy_Tales_Told_for_Children._First_Collection.\n", "pages": 784, "title": "Fairy tales", "year": 1836, "id": "2"}



In [19]:
with open("books.jsonl", "r", encoding="utf-8") as f:
    data = f.read()

result = client.collections["books"].documents.import_(
    data,
    {"action": "upsert"}
)

print(result)

{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":true}
{"success":tru

In [18]:
results = client.collections["books"].documents.search({
    "q": "*",
    "query_by": "title"
})

print("Documents found:", results["found"])

Documents found: 100


In [20]:
search_parameters = {
    "q": "machine learning",
    "query_by": "title,author,country,language"
}
client.collections["books"].documents.search(search_parameters)

{'facet_counts': [],
 'found': 0,
 'hits': [],
 'out_of': 100,
 'page': 1,
 'request_params': {'collection_name': 'books',
  'first_q': 'machine learning',
  'per_page': 10,
  'q': 'machine learning'},
 'search_cutoff': False,
 'search_time_ms': 0}

In [21]:
search_parameters={
    'q':'*',
    'query_by':'title',
    'facet_by':'country'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 10,
     'highlighted': 'United Kingdom',
     'value': 'United Kingdom'},
    {'count': 9, 'highlighted': 'Russia', 'value': 'Russia'},
    {'count': 9, 'highlighted': 'France', 'value': 'France'},
    {'count': 9, 'highlighted': 'United States', 'value': 'United States'},
    {'count': 5, 'highlighted': 'Italy', 'value': 'Italy'},
    {'count': 5, 'highlighted': 'Greece', 'value': 'Greece'},
    {'count': 5, 'highlighted': 'England', 'value': 'England'},
    {'count': 4, 'highlighted': 'Germany', 'value': 'Germany'},
    {'count': 3, 'highlighted': 'India', 'value': 'India'},
    {'count': 3, 'highlighted': 'Czechoslovakia', 'value': 'Czechoslovakia'}],
   'field_name': 'country',
   'sampled': False,
   'stats': {'total_values': 10}}],
 'found': 100,
 'hits': [{'document': {'author': 'Marguerite Yourcenar',
    'country': 'France/Belgium',
    'id': '100',
    'imageLink': 'images/memoirs-of-hadrian.jpg',
    'language': 'French',
    'link': 

In [24]:
### Langchain + Typesense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq


In [38]:
import os
from dotenv import load_dotenv

load_dotenv("notebook/.env")

print("Groq API key loaded:", os.getenv("GROQ_API_KEY") is not None)

Groq API key loaded: True


In [28]:
loader=TextLoader("test.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
docs=text_splitter.split_documents(documents)

embeddings=HuggingFaceEmbeddings()

d:\RAG\.venv\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nidhi Mehta\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1051.00it/s]


In [33]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        'host':'kqzi1b2675vwo8h3p-1.a1.typesense.net',
        'port':'443',
        'protocol':'https',
        'typesense_api_key':'bsL4IWBPL5jpxnqOWE44dJZzaj0Tpe6y',
        'typesense_collection_name':'lang-chain'
    

    },
)

In [34]:
query="What is artificial intelligence"
found_docs=docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence (AI) is a groundbreaking technology shaping the future of industries and everyday life. From advanced robotics and self-driving cars to virtual assistants and personalised recommendations, AI is revolutionising the way we work, live, and interact. This page consists of essays on artificial intelligence tailored to different lengths and needs, including 100, 150, 250, and 300-word formats. These essays provide a clear understanding of AIâ€™s transformative potential, challenges, and significance in the modern world. Whether you're looking for a short overview or a detailed analysis, this page covers it all.

Artificial Intelligence is about making machines that can think and learn like people. These machines are made to solve problems, make decisions, and do tasks like recognising voices, reading data, or driving cars.


In [35]:
### Retriever

retriever =docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000002419E7BA320>, search_kwargs={})

In [37]:
query="Artificial Intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content="Artificial Intelligence (AI) is a groundbreaking technology shaping the future of industries and everyday life. From advanced robotics and self-driving cars to virtual assistants and personalised recommendations, AI is revolutionising the way we work, live, and interact. This page consists of essays on artificial intelligence tailored to different lengths and needs, including 100, 150, 250, and 300-word formats. These essays provide a clear understanding of AIâ€™s transformative potential, challenges, and significance in the modern world. Whether you're looking for a short overview or a detailed analysis, this page covers it all.\n\nArtificial Intelligence is about making machines that can think and learn like people. These machines are made to solve problems, make decisions, and do tasks like recognising voices, reading data, or driving cars.")